# Notebook 5 – Forward Propagation

**Plan:** a tiny network **Input (2) → Hidden Layer (3 neurons) → Output Layer (1 neuron)**. We compute it **by hand with NumPy**, step by step, then rebuild the exact same network in **PyTorch** and confirm it gives the identical answer.

## Setup

In [1]:
import numpy as np
import torch
import torch.nn as nn


## 1. Forward Propagation

**Explain:** **Forward propagation** is the process of passing an input through a network, layer by layer, to produce a final prediction. Each layer does: (weighted sum + bias) → activation → pass to the next layer.

**Analogy:** An assembly line: raw material (input) enters at one end, each station (layer) transforms it a bit, and a finished product (prediction) comes out the other end.

**Implement:** define our fixed input and all the weights/biases we'll use for the whole notebook.

In [2]:
x = np.array([0.5, 0.8])                   
W1 = np.array([[0.2, -0.4, 0.1],              
               [0.3,  0.6, -0.5]])
b1 = np.array([0.1, -0.1, 0.05])
W2 = np.array([[0.7], [-0.3], [0.9]])         
b2 = np.array([0.2])
print("Input:", x)

Input: [0.5 0.8]


**Interpret:** These are exactly the ingredients forward propagation needs: one input, and a fixed set of weights/biases for each layer (already "trained" for this demonstration — training itself is covered in Notebook 6).


## 2. Input Layer

**Explain:** The **input layer** is simply the raw features themselves — no computation happens here, it just defines how many numbers enter the network.

**Analogy:** The raw material delivered to the start of the assembly line.

**Implement:**

In [3]:
print("Input layer size:", x.shape[0], "values:", x)

Input layer size: 2 values: [0.5 0.8]


**Interpret:** Our network starts with exactly 2 numbers — this fixes the shape of `W1` above (it must have 2 rows, one per input).


## 3. Weights and Biases

**Explain:** **Weights** scale each input's contribution to each neuron; **biases** shift each neuron's result independently of the inputs. Every connection between layers has its own weight; every neuron has its own bias.

**Analogy:** Weights are how much each ingredient counts in a recipe; bias is a fixed seasoning added regardless of the ingredients.

**Implement:**

In [4]:
print("W1 shape (inputs x hidden neurons):", W1.shape)
print("b1 shape (one bias per hidden neuron):", b1.shape)

W1 shape (inputs x hidden neurons): (2, 3)
b1 shape (one bias per hidden neuron): (3,)


**Interpret:** `W1` has shape (2, 3) — one weight for every (input, hidden neuron) pair — and `b1` has one bias per hidden neuron. This is exactly what lets each of the 3 hidden neurons look at the same 2 inputs differently.


## 4. Hidden Layer — Linear Transformation

**Explain:** The **linear transformation** at the hidden layer computes `z1 = x @ W1 + b1` — the weighted sum plus bias for every hidden neuron, all at once.

**Analogy:** Three separate judges (hidden neurons) each independently combining the same 2 scores using their own weightings.

**Implement:**

In [5]:
z1 = x @ W1 + b1
print("z1 (hidden layer, before activation):", z1.round(4))

z1 (hidden layer, before activation): [ 0.44  0.18 -0.3 ]


**Interpret:** 3 raw numbers, one per hidden neuron — each is a different weighted combination of the same 2 inputs. This is still purely linear; no decision-making non-linearity has been applied yet.


## 5. Hidden Layer — Activation

**Explain:** The **activation function** (ReLU here) is applied to `z1`, introducing non-linearity — without this step, stacking layers would be pointless (as shown in Notebook 4).

**Implement:**

In [6]:
def relu(v):
    return np.maximum(0, v)
a1 = relu(z1)
print("a1 (hidden layer, after ReLU):", a1.round(4))

a1 (hidden layer, after ReLU): [0.44 0.18 0.  ]


**Interpret:** Any negative value in `z1` became exactly 0; positive values passed through unchanged. This `a1` is the hidden layer's actual output — what gets passed on to the output layer.


## 6. Output Layer — Linear Transformation + Activation

**Explain:** The output layer repeats the same idea: `z2 = a1 @ W2 + b2`, then an activation (sigmoid, since this is a binary-style output) turns it into a final number between 0 and 1.

**Implement:**

In [7]:
def sigmoid(v):
    return 1 / (1 + np.exp(-v))
z2 = a1 @ W2 + b2
a2 = sigmoid(z2)
print("z2 (output layer, before activation):", z2.round(4))
print("a2 (output layer, after sigmoid):     ", a2.round(4))

z2 (output layer, before activation): [0.454]
a2 (output layer, after sigmoid):      [0.6116]


**Interpret:** The hidden layer's output (`a1`) became the *input* to this layer — this is the key idea of a multi-layer network: each layer's output feeds the next layer's input. The final sigmoid squashes everything into a clean 0–1 range.


## 7. Prediction

**Explain:** The **prediction** is simply the final output after the last layer's activation — the number we actually use (e.g., "70% probability of class 1").

**Implement:**

In [8]:
prediction = a2[0]
print(f"Final prediction: {prediction:.4f}  ->  predicted class: {int(prediction > 0.5)}")

Final prediction: 0.6116  ->  predicted class: 1


**Interpret:** After flowing through both layers, our 2 raw inputs became one clean probability, and thresholding at 0.5 gives a final class decision — this is the complete journey from input to prediction.


## Manual Forward Propagation — All Steps Together

In [ ]:
print("Input        x  :", x)
print("Hidden (lin) z1 :", z1.round(4))
print("Hidden (act) a1 :", a1.round(4))
print("Output (lin) z2 :", z2.round(4))
print("Output (act) a2 :", a2.round(4))
print("Prediction      :", round(prediction, 4))

**Interpret:** This is forward propagation end-to-end, entirely by hand: **input → linear → activation → linear → activation → prediction**. Every number here was produced with nothing but NumPy matrix multiplication and two simple functions.


## Same Network in PyTorch

**Implement:** build an identical architecture in PyTorch, load in the **exact same weights and biases**, and confirm we get the exact same prediction.

In [9]:
torch_model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1), nn.Sigmoid())
with torch.no_grad():
    torch_model[0].weight[:] = torch.tensor(W1.T, dtype=torch.float32)   # PyTorch stores weights transposed
    torch_model[0].bias[:] = torch.tensor(b1, dtype=torch.float32)
    torch_model[2].weight[:] = torch.tensor(W2.T, dtype=torch.float32)
    torch_model[2].bias[:] = torch.tensor(b2, dtype=torch.float32)
torch_prediction = torch_model(torch.tensor(x, dtype=torch.float32)).item()
print("NumPy prediction:  ", round(prediction, 4))
print("PyTorch prediction:", round(torch_prediction, 4))

NumPy prediction:   0.6116
PyTorch prediction: 0.6116


**Interpret:** Both give the **same prediction** — confirming that PyTorch's `nn.Linear` + `nn.ReLU` + `nn.Sigmoid` do exactly the same math we wrote by hand with NumPy. The only difference is that PyTorch also tracks gradients automatically and handles the matrix bookkeeping for us (and stores each layer's weight matrix transposed relative to how we wrote it manually).


## How Does an Input Finally Become a Prediction?

**Explain:** Tracing the full journey:
1. **Input layer** holds the raw numbers — no computation.
2. Each subsequent layer performs a **linear transformation** (`weights @ input + bias`), combining all inputs into new raw values.
3. An **activation function** is applied to those raw values, adding non-linearity and producing that layer's actual output.
4. That output becomes the **input to the next layer** — steps 2–3 repeat for every layer.
5. The **final layer's** activation produces the **prediction**, already in the format the task needs (e.g. a 0–1 probability).

So an input "becomes" a prediction through a **chain of linear transformations and non-linear activations**, each one reshaping the data a little further, until it reaches a form that directly answers the question being asked.

**Implement:** visualize the chain as a simple diagram using the shapes from this notebook.

In [10]:
shapes = [("Input", x.shape), ("Hidden (z1, a1)", z1.shape), ("Output (z2, a2)", z2.shape)]
for name, shape in shapes:
    print(f"{name:18} shape {shape}")
print("\nFlow: Input -> [Linear+Bias -> Activation] -> Hidden -> [Linear+Bias -> Activation] -> Prediction")

Input              shape (2,)
Hidden (z1, a1)    shape (3,)
Output (z2, a2)    shape (1,)

Flow: Input -> [Linear+Bias -> Activation] -> Hidden -> [Linear+Bias -> Activation] -> Prediction


**Interpret:** The data's *shape* changes at every stage (2 numbers → 3 numbers → 1 number), but the *process* is identical each time: multiply by weights, add bias, apply activation. Repeating this simple two-step recipe, layer after layer, is the entire mechanism behind how any neural network — no matter how large — turns an input into a prediction.